# 05 — Independent validation across 1,611 increments per precision

**Goal.** Reproduce the numerical validation summarized in *artigo_ulp.pdf*, Section 3, and the code appendix of *livro_previa.pdf*: 1,001 odd integers, 300 dyadic values, and 310 intended non-dyadic values, **for each of binary64 and binary32**.

**Important distinction.** Passing this notebook means that *these samples* agree with the model **on its stated domain**; it is not a formal proof for every real number, nor does it establish platform-independent bitwise behavior outside the stated arithmetic assumptions.

In [1]:
from pathlib import Path
import sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'ulp_lab.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'ulp_lab.py').exists():
    raise FileNotFoundError('Open Jupyter in the extracted ULP_Jupyter_Reproducibility folder.')
sys.path.insert(0, str(ROOT / 'src'))
from ulp_lab import (evaluate, ulp, v2, floor_log2, theory, possible_results,
                     sample_binade, valid_same_binade_mask, generate_cases, validate_one)
FIG = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
RES = ROOT / 'results'; RES.mkdir(exist_ok=True)
BLUE='#2367A6'; ORANGE='#DD8A29'; GREEN='#2C977F'; RED='#B85751'; PURPLE='#8572AB'
plt.rcParams.update({'font.size':10,'figure.dpi':110,'savefig.dpi':190,
                     'axes.grid':True,'grid.alpha':.20,
                     'axes.spines.top':False,'axes.spines.right':False})
print('Working directory:', ROOT)
print('NumPy:', np.__version__)

Working directory: /mnt/data/ULP_Jupyter_Reproducibility
NumPy: 2.3.5


## Validation protocol and reproducibility configuration

1. Create a deterministic family of 1,611 inputs (seed 42). The same list is tested in both precisions, rather than consuming fresh random constants separately as the book's script does; this makes the two precisions directly comparable.
2. For each `c`, calculate the predicted binades $E_1,\ldots,E_2$.
3. Draw `SAMPLES_PER_BINADE` uniformly sampled `x` values in each test binade and recompute `y=fl(x**10)`.
4. Keep points whose **calculated** `y` belongs to the intended binade, satisfies `y >= c`, and does not cross the binade upper boundary on addition. Report how many samples were excluded.
5. Compare each observed `f` with the model's permitted set; separately check that $|f-c|\le0.5\,\mathrm{ULP}(y)$ for included points.

Set `SAMPLES_PER_BINADE=3000` for the sampling scale used in the paper's validation script. The default **1500** is the low end of the paper's stated 1,500–4,000 range; it still checks all 1,611 values per precision.

In [2]:
SEED = 42
SAMPLES_PER_BINADE = 1500
EXPONENT_N = 10
case_groups=generate_cases(SEED)
print('Input counts:',{k:len(v) for k,v in case_groups.items()})
assert sum(map(len,case_groups.values()))==1611
print('Number of values per precision:',1611)

Input counts: {'odd integers': 1001, 'dyadic fractions': 300, 'non-dyadic examples': 310}
Number of values per precision: 1611


## Execute the full validation

The checks use NumPy in each dtype, never rely on symbolic cancellation, and use `np.longdouble` **only** to detect possible binade crossings without modifying the evaluated expression.

In [3]:
import time
rng=np.random.default_rng(SEED)
rows=[];examples=[]
start=time.perf_counter()
for dtype in [np.float64,np.float32]:
    for family,cs in case_groups.items():
        for idx,c in enumerate(cs):
            test=validate_one(c,dtype=dtype,n=EXPONENT_N,
                              samples=SAMPLES_PER_BINADE,rng=rng)
            rows.append({'dtype':dtype.__name__,'family':family,'index':idx,'c':c,
                         'pass':bool(test['pass']),'checked samples':test['examined'],
                         'excluded crossing':test['excluded_edge'],
                         'excluded y<c':test['excluded_y_lt_c'],
                         'tie outcomes seen':test['tie_variants_seen']})
            if not test['pass']:
                examples.append((dtype.__name__,family,c,test['mismatches']))
validation=pd.DataFrame(rows)
print(f'Elapsed: {time.perf_counter()-start:.2f} seconds')
summary=validation.groupby(['dtype','family'],sort=False).agg(
    passed=('pass','sum'),cases=('pass','size'),
    checked=('checked samples','sum'),
    crossing_exclusions=('excluded crossing','sum'),
    y_lt_c_exclusions=('excluded y<c','sum')).reset_index()
display(summary)
validation.to_csv(RES/'05_validation_per_increment.csv',index=False)
summary.to_csv(RES/'05_validation_summary.csv',index=False)
if examples:
    print('First discrepancies:',examples[:5])
assert len(validation)==3222
assert validation['pass'].all(),f'Unexpected model discrepancy: {examples[:3]}'

Elapsed: 4.74 seconds


,dtype,family,passed,cases,checked,crossing_exclusions,y_lt_c_exclusions
0,float64,odd integers,1001,1001,20988000,0,0
1,float64,dyadic fractions,300,300,4081500,0,0
2,float64,non-dyadic examples,310,310,24950331,522669,0
3,float32,odd integers,1001,1001,20987428,564,0
4,float32,dyadic fractions,300,300,4081483,14,0
5,float32,non-dyadic examples,310,310,11427122,514371,0


## Verify the paper's key numerical conclusions separately

In [4]:
checks=[
  ('Common onset for c=1,3,101 (binary64)',
      len({theory(c).x1 for c in [1,3,101]})==1),
  ('c=101 has L=7 and x2=64 (binary64)',
      theory(101).L==7 and theory(101).x2==64),
  ('c=3/4 has L=2 (binary64)',theory(3/4).L==2),
  ('c=5/16 has L=3 (binary64)',theory(5/16).L==3),
  ('c=0.1 has L=52 (binary64)',theory(0.1).L==52),
  ('c=1/3 has L=53 (binary64)',theory(1/3).L==53),
  ('c=0.1 has L=24 (binary32)',theory(0.1,np.float32).L==24),
  ('c=1 has L=1 (both formats)',
      theory(1).L==1 and theory(1,np.float32).L==1),
]
for label,passed in checks:
    print(('PASS' if passed else 'FAIL')+' | '+label)
assert all(passed for _,passed in checks)

PASS | Common onset for c=1,3,101 (binary64)
PASS | c=101 has L=7 and x2=64 (binary64)
PASS | c=3/4 has L=2 (binary64)
PASS | c=5/16 has L=3 (binary64)
PASS | c=0.1 has L=52 (binary64)
PASS | c=1/3 has L=53 (binary64)
PASS | c=0.1 has L=24 (binary32)
PASS | c=1 has L=1 (both formats)


## Interpretation and limitations

- **Result reported in the sources:** 1,611 increments passed the validation per precision.
- **Result reproduced here:** the observed pass/fail totals printed above, based on the generated constants and random samples. Check the totals rather than assuming they must match.
- **Exceptions:** points where `y<c` or `y+c` crosses a binade are counted and **excluded** from the proposition's test. The paper discusses these regions separately; there can be errors of up to about one ULP in boundary crossings.
- **Model choices:** all `c` are cast to the tested dtype before arithmetic; mathematical non-dyadic constants become finite dyadics on storage, so their $\nu$ and $L$ are determined from the stored bits.
- **Platform sensitivity:** NumPy, compiler implementations of `np.power`, rounding mode and processor differences can move a few numerical transition points; random seeds ensure deterministic pseudorandom samples in a fixed software environment.

The other five notebooks regenerate **all nine figures** used in the English LaTeX report, and write their corresponding CSV results under `results/`.